# 13.4 사라지는 적분

In [1]:
import numpy as np

c2v = {
    "A1": [1, 1, 1, 1],
    "A2": [1, 1, -1, -1],
    "B1": [1, -1, 1, -1],
    "B2": [1, -1, -1, 1],
}
c2v_sizes = [1, 1, 1, 1]
c3v = {
    "A1": [1, 1, 1],
    "A2": [1, 1, -1],
    "E": [2, -1, 0],
}
c3v_sizes = [1, 2, 3]


def reduce(chi, table, sizes):
    return {
        name: round(sum(g * a * b for g, a, b in zip(sizes, chi, row)) / sum(sizes))
        for name, row in table.items()
    }


def direct_product(table, *names):
    chi = np.ones(len(next(iter(table.values()))), dtype=int)
    for name in names:
        chi = chi * np.array(table[name])
    return [int(v) for v in chi]


for pair in [("B1", "B1"), ("B1", "B2"), ("A1", "B2")]:
    chi = direct_product(c2v, *pair)
    print(
        f"C2v  {pair[0]:>2} x {pair[1]:<2}: 지표 {chi} -> {reduce(chi, c2v, c2v_sizes)}"
    )

print()
for pair in [("E", "E"), ("A1", "E")]:
    chi = direct_product(c3v, *pair)
    print(
        f"C3v  {pair[0]:>2} x {pair[1]:<2}: 지표 {chi} -> {reduce(chi, c3v, c3v_sizes)}"
    )

C2v  B1 x B1: 지표 [1, 1, 1, 1] -> {'A1': 1, 'A2': 0, 'B1': 0, 'B2': 0}
C2v  B1 x B2: 지표 [1, 1, -1, -1] -> {'A1': 0, 'A2': 1, 'B1': 0, 'B2': 0}
C2v  A1 x B2: 지표 [1, -1, -1, 1] -> {'A1': 0, 'A2': 0, 'B1': 0, 'B2': 1}

C3v   E x E : 지표 [4, 1, 0] -> {'A1': 1, 'A2': 1, 'E': 1}
C3v  A1 x E : 지표 [2, -1, 0] -> {'A1': 0, 'A2': 0, 'E': 1}


In [4]:
r, theta = 0.9572 * 1.8897, np.radians(104.52)  # 원자 단위계
O = np.array([0, 0, 0])
H1 = np.array([0, r * np.sin(theta / 2), -r * np.cos(theta / 2)])
H2 = np.array([0, -r * np.sin(theta / 2), -r * np.cos(theta / 2)])

# 원점에 대해 대칭인 3차원 격자
g = np.linspace(-5, 5, 100)
dV = (g[1] - g[0]) ** 3
X, Y, Z = np.meshgrid(g, g, g, indexing="ij")


def gauss(a, c):
    return np.exp(-a * ((X - c[0]) ** 2 + (Y - c[1]) ** 2 + (Z - c[2]) ** 2))


def dist(c):
    return np.sqrt((X - c[0]) ** 2 + (Y - c[1]) ** 2 + (Z - c[2]) ** 2 + 0.01)


V = -8 / dist(O) - 1 / dist(H1) - 1 / dist(H2)
h = g[1] - g[0]


def laplacian(f):
    return sum(np.gradient(np.gradient(f, h, axis=k), h, axis=k) for k in range(3))


def apply_H(f):
    return -0.5 * laplacian(f) + V * f


orbitals = {
    "O 1s": gauss(7.0, O),
    "O 2s": gauss(0.9, O) - 0.3 * gauss(7.0, O),
    "O 2pz": Z * gauss(0.9, O),
    "H+": gauss(0.5, H1) + gauss(0.5, H2),
    "O 2px": X * gauss(0.9, O),
    "O 2py": Y * gauss(0.9, O),
    "H-": gauss(0.5, H1) - gauss(0.5, H2),
}
species = ["A1", "A1", "A1", "A1", "B1", "B2", "B2"]

names = list(orbitals)
F = [orbitals[k] for k in names]
norm = np.sqrt([np.sum(f * f) * dV for f in F])
HF = [apply_H(f) for f in F]
M = np.array([[np.sum(a * Hb) * dV for Hb in HF] for a in F]) / np.outer(norm, norm)

print(" " * 12 + "".join(f"{n:>7}" for n in names))
for name, sp, row in zip(names, species, M):
    marks = "".join(f"{'x' if abs(v) > 1e-6 else '.':>7}" for v in row)
    print(f"{name:>6} ({sp}){marks}")

upper = np.triu_indices(len(F))
print()
print("계산해야 할 행렬 요소:", len(upper[0]), "개")
print("0의 갯수:", np.sum(np.abs(M[upper]) < 1e-6), "개")

               O 1s   O 2s  O 2pz     H+  O 2px  O 2py     H-
  O 1s (A1)      x      x      x      x      .      .      .
  O 2s (A1)      x      x      x      x      .      .      .
 O 2pz (A1)      x      x      x      x      .      .      .
    H+ (A1)      x      x      x      x      .      .      .
 O 2px (B1)      .      .      .      .      x      .      .
 O 2py (B2)      .      .      .      .      .      x      x
    H- (B2)      .      .      .      .      .      x      x

계산해야 할 행렬 요소: 28 개
0의 갯수: 14 개
